In [5]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import plotly.express as px
from seaborn import heatmap
from sklearn.cluster import HDBSCAN

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder
)

In [62]:
from sklearn.utils.validation import check_random_state
check_random_state(42)

RandomState(MT19937) at 0x73ABCBC1E540

In [4]:
DATA_DIR = '../../data/raw'
DATASET = f'{DATA_DIR}/dataset_practica_final.csv'

In [34]:
dset = pd.read_csv(DATASET)
dset_cp = dset.copy()

In [35]:
dset_cp['country'] = dset_cp['country'].fillna('UNKNOWN')

In [48]:
country_frequency = pd.DataFrame(dset['country'].value_counts(normalize=True)).reset_index()

In [52]:
dset_cp = dset_cp.merge(country_frequency, on='country')

/tmp/ipykernel_5487/451231498.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dset_cp[dset_cp.proportion < 0.05]['country'] = 'OTROS'


In [ ]:
def loader(dataset_csv_filename:str, OHE:bool) -> (pd.core.frame.DataFrame, pd.core.frame.DataFrame, pd.core.series.Series, pd.core.series.Series):
    '''
    dataset_csv_filename: ruta al csv con los datos
    OHE: si es true, los campos categóricos se condifican con OneHotEncoder. 
         si false los campos de texto se codifican a categoría y después a numérico usando sus códigos
    '''
    # se setea random_state para que los resultados sean predecibles (que el resultado sea el mismo siempre)
    check_random_state(42)

    # carga del csv en un dataframe
    dset = pd.read_csv(DATASET)
    
    # Eliminación de variables con fugas de datos (Data Leakage)
    columns_to_drop =[
                        'reservation_status',
                        'reservation_status_date',
                        'arrival_date_year',
                        'assigned_room_type']
    dset.drop(columns_to_drop, axis= 1, inplace=True)

    # --- LIMPIEZA DE ANOMALÍAS ---
    
    # Reemplazar el valor negativo por la mediana
    dset.loc[dset['adr'] < 0, 'adr'] = dset['adr'].median()

    # Eliminar las columnas con 0 adultos
    dset = dset[dset['adults'] != 0]

    # Eliminar las filas con 10 niños y 10 bebes
    dset = dset[dset['children'] != 10]
    dset = dset[dset['babies'] != 10]

    # --- LIMPIEZA DE ANOMALÍAS ---


    # Agrupar países con poca representación
    top_countries = dset['country'].value_counts().nlargest(10).index
    dset['country'] = dset['country'].where(dset['country'].isin(top_countries), 'Other')

    # Convertir el mes a valor numérico para mantener la temporalidad
    month_map = {'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
                 'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12}
    dset['arrival_date_month'] = dset['arrival_date_month'].map(month_map)

    # Simplificar Agente y Compañía a variables binarias
    dset['has_company'] = dset['company'].notnull().astype(int)
    dset['has_agent'] = dset['agent'].notnull().astype(int)
    dset.drop(['company', 'agent'], axis=1, inplace=True)
    
    # relleno de los valores faltantes
    dset['children'] = dset['children'].fillna(dset['children'].median())

    # separación en target (variable objetivo) y features
    y = dset['is_canceled']
    X = dset.drop(['is_canceled'], axis=1)

    # train_test_split se hace antes de trasnformar nada
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
    

    # OneHot encoder de las variables categóricas (no ordinales) y standa
    list_one_hot_cols = []
    if OHE:
        list_one_hot_cols = [
                            'hotel',
                            'meal',
                            'country',
                            'market_segment',
                            'distribution_channel',
                            #'is_repeated_guest',
                            'reserved_room_type',
                            #'assigned_room_type',
                            'deposit_type',
                            #'agent',
                            #'company',
                            'customer_type']
    else:
        # primero se ponen en una lista los cmps de texto
        # después se convierten los campos a catgorías, se toman sus códigos numéricos (.cat.codes) y se sustitye a los string originales por ellos
        text_fields = [c for c in X_train.columns if X_train[c].dtype==object]
        for c in text_fields:
            X_train[c] = X_train[c].astype('category').cat.codes
            X_test[c] = X_test[c].astype('category').cat.codes  

    # Mejor hacer la lista dinámica, para que no se pierda nada
    list_numeric_cols = [col for col in X_train.columns if col not in list_one_hot_cols]

    columns_preprocessor = ColumnTransformer(transformers=[
        ('one_hot', OneHotEncoder(drop='first', sparse_output=False), list_one_hot_cols),
        #('ordinal', OrdinalEncoder(), list_ordinal_cols),
        ('numeric', MinMaxScaler(), list_numeric_cols)
    ])

    # fit_transform solo en train, transform en test
    X_train_processed = columns_preprocessor.fit_transform(X_train)
    X_test_processed = columns_preprocessor.transform(X_test)

    # Restaurar a DataFrame para no perder los nombres
    nombres_columnas = columns_preprocessor.get_feature_names_out()

    # Se limpia el texto que añade scikit-learn por defecto
    nombres_columnas = [nombre.replace('one_hot__', '').replace('numeric__', '') for nombre in nombres_columnas]

    X_train = pd.DataFrame(X_train_processed, columns=nombres_columnas, index=X_train.index)
    X_test = pd.DataFrame(X_test_processed, columns=nombres_columnas, index=X_test.index)

    return (X_train, X_test, y_train, y_test)

In [ ]:
X_train, X_test, y_train, y_test = loader(DATASET, False)